# 🖋️ 04 — Signatures that contracts believe

*How Bell makes a promise that anyone can check, nobody can forge, and a Solidity
contract will accept — byte for byte.*

Chapter 03 had one piece of cardboard in it. Every robbery you ran ended at a real
check — except forgery, where the toy compared a `signed_by` field and we admitted
that proved nothing. Meanwhile, in Act IV, the *real* contract verified a *real*
signature and rejected your tampered offer with `BadSignature` — magic you used but
didn't open. Today you open it: by the end of this notebook you will have **built the
exact 32-byte digest the contract builds, by hand, layer by layer**, signed it with
Bell's key, and proven your bytes equal the contract's bytes — not "conceptually",
but `==`.

**You need:** nothing for §0–§3 (pure Python). §4 (asking the *live* contract to
referee your hand-built digest) wants [Foundry](https://getfoundry.sh) and
`forge build --root contracts` — without them those cells politely skip.

**How to work through it:** run every cell in order; on a **✏️ Your turn**, edit the
scaffold *before* opening the solution. 🧭 Decision boxes and the closing 📝 section
work as in chapter 03: honest registers, paper coordinates.

## 0 · The story so far, and today's question

Where chapter 03 left the vending machine: it refuses stale quotes (`OfferExpired`),
quotes meant for someone else (`WrongConsumer`), and second redemptions
(`OfferAlreadyUsed`). But its forgery check was:

```python
if offer.signed_by != offer.provider:            # cardboard; ch. 04 makes it real
    raise Exception("BadSignature")
```

`signed_by` is just a string in a dataclass. Mallory writes `signed_by="Bell"` and the
toy is fooled. What the machine actually needs is a thing with three properties at once:

1. **only Bell can produce it** (else it proves nothing),
2. **anyone can check it** (the contract, Ada, a judge — without Bell's help),
3. **it covers every byte of the offer** (change the price, and it dies).

That thing is a **digital signature**, and the asymmetry in (1)+(2) is the whole trick:
one secret key that signs, one public identity that verifies. Chapter 01 gave you the
key/address pair; today the pair goes to work.

## 1 · Signatures from zero

Bell's identity, since chapter 01, is a key pair: a secret number (the private key) and
an address derived from it. Signing takes *the secret* plus *a message* and produces 65
bytes. Verifying — and this is the part that feels like a magic trick — takes only the
message and the 65 bytes, and **recovers the address that must have signed**. No secret
needed to check; the secret can never be computed back out of the signature.

Watch it work on a plain sentence first (these are anvil's well-known dev keys — public
lab constants, the same cast as always):

In [ ]:
from eth_account import Account
from eth_account.messages import encode_defunct

from a2a_interfaces import fixtures as fx
from chainmcp.testing import ANVIL_KEYS

bell_key = ANVIL_KEYS["bell"]

message = encode_defunct(text="Bell promises 50 Mbps, 14:00-16:00, for 10 TOK")
signed = Account.sign_message(message, bell_key)

print("the promise    :", "Bell promises 50 Mbps, 14:00-16:00, for 10 TOK")
print("the signature  :", "0x" + signed.signature.hex()[:40] + "…", f"({len(signed.signature)} bytes)")

recovered = Account.recover_message(message, signature=signed.signature)
print("recovered signer:", recovered)
print("Bell's address  :", fx.BELL)
print("match?          :", recovered == fx.BELL)

Verification never touched `bell_key`. Anyone holding just the message and the 65 bytes
gets Bell's address back out. Now the property that makes signatures *useful*: tamper
with **one character** of the message and see what the same signature recovers to:

In [ ]:
tampered = encode_defunct(text="Bell promises 50 Mbps, 14:00-16:00, for 1 TOK")   # 10 → 1

recovered = Account.recover_message(tampered, signature=signed.signature)
print("recovered signer:", recovered)
print("Bell's address  :", fx.BELL)
print("match?          :", recovered == fx.BELL)

No error, no warning — just a **stranger's address**. That's the design: a signature
over message *M* is only Bell's signature over exactly *M*. Verify it against anything
else and the math produces some other address, one nobody holds the key to. This is
what chapter 03's tamper exercise ran into: you changed the price after signing, the
contract recovered a stranger, `BadSignature`.

Under the hood the 65 bytes are three numbers — `r`, `s` (two coordinates from the
elliptic-curve math) and `v` (one byte of disambiguation):

In [ ]:
print("r =", hex(signed.r))
print("s =", hex(signed.s))
print("v =", signed.v)
print("total: 32 + 32 + 1 =", len(signed.signature), "bytes")

> **🧭 Decision (pragmatic) — the cryptography is a black box, on purpose**
>
> The curve (secp256k1), the hash (keccak-256), the recovery trick — all inherited,
> unmodified, from Ethereum's standard toolbox; verification on the contract side is
> OpenZeppelin's audited `ECDSA.recover`, not our code. One *could* study or vary the
> primitives; we don't, and not because they're uninteresting: **cryptographic design
> is not this project's contribution, and rolling your own is how projects die.** What
> this project *does* contribute is one layer up: deciding exactly *which bytes* get
> signed — and that's where the rest of this notebook lives.
> **In the paper:** nothing to defend in Design; one implicit line in §5.2 (standard
> ERC-721 + EIP-712 + ECDSA components).

**✏️ Your turn 1 — Ada signs too**

Sign the message `"Ada accepts"` with **Ada's** key (`ANVIL_KEYS["ada"]`), recover the
signer, and write your prediction of the recovered address (you've known it since
chapter 01) in a comment *before* running. Then check it against `fx.ADA`.

In [ ]:
# my prediction: 0x...
# ada_signed = ...
# recovered  = ...

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
ada_signed = Account.sign_message(encode_defunct(text="Ada accepts"), ANVIL_KEYS["ada"])
recovered = Account.recover_message(encode_defunct(text="Ada accepts"),
                                    signature=ada_signed.signature)
print(recovered, "==", fx.ADA, "→", recovered == fx.ADA)
```

The prediction is `0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266` — Ada's address falls
out of Ada's key, always. One key, one address, one signer identity.

</details>

## 2 · The ambiguity robberies — why "just sign the offer" fails

So Bell should… sign the offer. Obviously. But an offer is a *structure* — twelve
fields — and a signature covers *bytes*. Someone must decide how the structure becomes
bytes, and that decision is a security surface. Two robberies to prove it — both, note,
robberies against **honest** parties; no Mallory required.

**Robbery A — the same offer, serialized twice.** The obvious encoding is JSON. Bell's
laptop serializes the offer one way and signs it. Ada's laptop — different library,
different habits — re-serializes the *same* offer to check the signature:

In [ ]:
import json

offer = {"provider": "Bell", "capacity_mbps": 50, "price": 10}

bell_bytes = json.dumps(offer).encode()                      # Bell's laptop
ada_bytes  = json.dumps(offer, sort_keys=True, separators=(",", ":")).encode()  # Ada's

print("Bell signed :", bell_bytes)
print("Ada checks  :", ada_bytes)
print("same offer? yes.  same bytes?", bell_bytes == ada_bytes)

signed = Account.sign_message(encode_defunct(bell_bytes), bell_key)
recovered = Account.recover_message(encode_defunct(ada_bytes), signature=signed.signature)
print("Ada recovers:", recovered, "→ Bell?", recovered == fx.BELL)

Same meaning, different bytes, verification fails — between two *honest* machines. A
signature scheme that breaks when nobody cheated is worse than useless: it trains
everyone to ignore failures.

**Robbery B — two different offers, identical bytes.** Fine, says a hasty engineer,
skip JSON: concatenate the fields and sign that. Watch what concatenation does to the
boundary between fields:

In [ ]:
offer_1 = {"provider": "Bell1", "price": "05"}    # provider "Bell1", price  5
offer_2 = {"provider": "Bell10", "price": "5"}    # provider "Bell10", price 5

bytes_1 = (offer_1["provider"] + offer_1["price"]).encode()
bytes_2 = (offer_2["provider"] + offer_2["price"]).encode()

print("offer_1 encodes to:", bytes_1)
print("offer_2 encodes to:", bytes_2)
print("different offers, same bytes?", bytes_1 == bytes_2)
print("→ one signature is now valid for BOTH offers.")

A signature over `b"Bell105"` is a promise about… which offer? Both. Neither. The
encoding erased the field boundaries, so the signature can't pin down what was
promised.

The lesson, and it's the entire reason the next section exists:

> a signature is only as unambiguous as the **encoding** of what it signs. You must
> sign a *canonical* (exactly one way to write it) encoding of *typed* (field
> boundaries and types baked in) data.

Ethereum's standard answer is **EIP-712** — "typed structured data signing". It looks
fussy. Every ounce of the fussiness is one of these two robberies, pre-empted. Let's
build it with our own hands.

## 3 · EIP-712 by hand — five layers to the digest

The goal: reduce Bell's twelve-field offer to **one 32-byte digest** such that (a) the
same offer always produces the same digest, on any machine, in any language, and (b)
any change to any field produces a different digest. Then Bell signs the digest.

We'll build it in five layers, with real bytes visible at each step, using the story's
canonical offer (`fx.CANONICAL_OFFER` — the exact object chapter 03 settled).

### Layer 1 — declare the type, hash the declaration

First, the field boundaries that Robbery B erased: EIP-712 starts from a **type
string** — the struct's name, fields, and types, in order, in one exact spelling. This
is quoted from `contracts/src/Settlement.sol`, where it defines `OFFER_TYPEHASH`:

In [ ]:
from eth_utils import keccak

TYPE_STRING = (
    "Offer(address provider,address consumer,uint8 serviceType,bytes32 resourceId,"
    "bytes params,uint64 startTime,uint64 endTime,address paymentToken,uint256 price,"
    "uint64 validUntil,bytes32 salt,bytes32 termsHash)"
)

OFFER_TYPEHASH = keccak(text=TYPE_STRING)
print("OFFER_TYPEHASH =", "0x" + OFFER_TYPEHASH.hex())

That hash is the fingerprint of the *shape* — it will be mixed into every offer digest,
so a signature can never be confused between two structs that happen to hold similar
values. And it is spelling-sensitive on purpose:

In [ ]:
# The same twelve fields with just TWO WORDS swapped (price before paymentToken):
reordered = TYPE_STRING.replace(
    "address paymentToken,uint256 price", "uint256 price,address paymentToken"
)

print("original  :", "0x" + keccak(text=TYPE_STRING).hex()[:32] + "…")
print("reordered :", "0x" + keccak(text=reordered).hex()[:32] + "…")
print("same universe?", keccak(text=TYPE_STRING) == keccak(text=reordered))

One reordered word and every signature in the system recovers to a stranger. This is
why the repo pins the field order in *three* places that must agree byte for byte: the
Solidity type string, the Python `OFFER_TYPES` dict, and `docs/03-interfaces.md` §1.4 —
and why a cross-stack test exists just to catch drift between them.

### Layer 2 — encode the values on a fixed grid

Robbery B died of missing boundaries; the cure is a **fixed grid**: every field gets
exactly 32 bytes (Solidity's `abi.encode`). Numbers are right-aligned in their slot,
addresses left-padded — no delimiters to fake, because position *is* the delimiter.

One rule needs calling out before we encode, because it bites everyone (the contract's
own comment says so): `params` is a **dynamic** type (`bytes`, any length — it holds
the ABI-encoded service parameters, 64 bytes for bandwidth). A variable-length field
would wreck the fixed grid, so EIP-712 says: **dynamic values enter the grid as their
keccak hash** — 32 bytes, always, and still tamper-evident.

In [ ]:
import eth_abi

offer = fx.CANONICAL_OFFER
wire = offer.model_dump(by_alias=True)     # the camelCase wire shape (docs/03 §1.4)

encoded = eth_abi.encode(
    ["bytes32",                      # the typehash — layer 1, always slot 0
     "address", "address", "uint8", "bytes32",
     "bytes32",                      # ← params: keccak(bytes), per the dynamic rule
     "uint64", "uint64", "address", "uint256", "uint64", "bytes32", "bytes32"],
    [OFFER_TYPEHASH,
     wire["provider"], wire["consumer"], wire["serviceType"],
     bytes.fromhex(wire["resourceId"][2:]),
     keccak(bytes.fromhex(wire["params"][2:])),          # the rule, applied
     wire["startTime"], wire["endTime"], wire["paymentToken"],
     int(wire["price"]), wire["validUntil"],
     bytes.fromhex(wire["salt"][2:]), bytes.fromhex(wire["termsHash"][2:])],
)

print(f"{len(encoded)} bytes = 13 slots × 32.  The grid, one row per slot:\n")
names = ["typehash", "provider", "consumer", "serviceType", "resourceId",
         "keccak(params)", "startTime", "endTime", "paymentToken", "price",
         "validUntil", "salt", "termsHash"]
for name, i in zip(names, range(0, len(encoded), 32)):
    print(f"{name:15}{encoded[i:i + 32].hex()}")

Read the grid for a moment — it's the whole offer, laid out like a form. You can *see*
Bell's address right-padded into its slot, the price `8ac7230489e80000` (10·10¹⁸ in
hex), the window timestamps, ticket #7's resource id. Exactly one way to write this
offer; no two offers write the same grid.

### Layer 3 — hash the grid

The **struct hash**: one keccak over the grid. This is the 32-byte identity of *this
offer* — the value chapter 03's `consumed` ledger keys by (remember "the identity of a
promise is the promise itself"? Here's the machinery).

In [ ]:
struct_hash = keccak(encoded)
print("structHash =", "0x" + struct_hash.hex())

### Layer 4 — say *where* this promise lives: the domain

A subtle replay hole remains. Suppose a second marketplace deploys the *same* contract
code, or a test chain carries the same offer. A digest built only from the offer's
fields would be valid **everywhere at once** — sign once on the test chain, Mallory
redeems it on the real one.

EIP-712's answer is the **domain separator**: a hash of *where* signatures are meant
to be checked — protocol name, version, chain id, and the verifying contract's
address — mixed into every digest.

In [ ]:
DOMAIN_TYPE = "EIP712Domain(string name,string version,uint256 chainId,address verifyingContract)"

CHAIN_ID = 31337                                              # anvil's chain id
SETTLEMENT = "0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512"     # deterministic: same
# deployer + same deploy order on a fresh chain ⇒ same address, every lab run (ch. 01)

domain_separator = keccak(eth_abi.encode(
    ["bytes32", "bytes32", "bytes32", "uint256", "address"],
    [keccak(text=DOMAIN_TYPE),
     keccak(text="A2AProvisioning"),      # the protocol's name…
     keccak(text="0"),                    # …and version — pinned in docs/03 §2.1
     CHAIN_ID, SETTLEMENT],
))
print("domainSeparator =", "0x" + domain_separator.hex())

> **🧭 Decision (principled) — sign typed data under a domain, not a JSON blob**
>
> **Chosen:** offers are signed as EIP-712 typed structs under the pinned domain
> `("A2AProvisioning", "0", chainId, settlement address)`.
> **Alternatives:** (a) `personal_sign` over a JSON serialization of the offer;
> (b) sign the bare struct hash, no domain.
> **Why:** (a) is Robbery A — serialization isn't canonical across machines, and the
> contract would have to parse JSON on-chain (absurdly expensive) or trust a hash of
> bytes it can't interpret. (b) re-opens replay one level up: the same signed struct
> would verify on every chain and every contract that shares the struct layout —
> domain binding is what makes a signature a promise *here*, to *this* contract, and
> worthless anywhere else. EIP-712 also has a side benefit you've seen if you own a
> wallet: it's what lets wallets *display* what's being signed, field by field, instead
> of an opaque hex blob.
> **Cost:** the fussiness you're living through right now — a five-layer encoding and
> three codebases that must agree byte for byte (that's a real maintenance tax; the
> repo pays it with a dedicated cross-stack test).
> **In the paper:** §4.4 (the offer struct and its EIP-712 signature) — and it
> pre-answers the reviewer question "what stops cross-chain or cross-contract replay?"

### Layer 5 — the final digest

Prefix `0x19 0x01` (a marker meaning "this is EIP-712 material, nothing else"), then
domain, then struct hash — one last keccak:

In [ ]:
digest = keccak(b"\x19\x01" + domain_separator + struct_hash)
print("digest =", "0x" + digest.hex())
print("\nThese 32 bytes ARE the offer, as far as any signature is concerned.")

Now Bell signs the digest — and we check our five hand-built layers against the
repo's production signer (`chainmcp.signing`), which must produce the *same* bytes
from the same offer, or every deal in the system dies at `BadSignature`:

In [ ]:
from chainmcp.signing import offer_digest as production_digest

hand = digest
prod = production_digest(fx.CANONICAL_OFFER, CHAIN_ID, SETTLEMENT)

print("hand-built :", "0x" + hand.hex())
print("production :", "0x" + prod.hex())
print("byte-for-byte equal?", hand == prod)

signed_digest = Account.unsafe_sign_hash(hand, bell_key)    # "unsafe" = raw-hash signing:
# fine HERE because we built the hash ourselves; wallets refuse blind hashes for a reason.
recovered = Account._recover_hash(hand, signature=signed_digest.signature)
print("\nBell signs the digest; recovery gives:", recovered)
print("Bell?", recovered == fx.BELL)

**✏️ Your turn 2 — one salt bit, whole new promise**

Chapter 03 said two same-terms offers are distinguished by their salt. Prove it at the
byte level: take `fx.CANONICAL_OFFER`, change only the salt
(`offer.model_copy(update={"salt": "0x" + "77" * 32})`), and rerun the production
digest. Predict first: how many of the 32 digest bytes change — one, a few, or
basically all of them?

In [ ]:
# prediction: ... of 32 bytes change
# salted = fx.CANONICAL_OFFER.model_copy(update={...})
# ...compare production_digest(...) for both...

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
salted = fx.CANONICAL_OFFER.model_copy(update={"salt": "0x" + "77" * 32})
d1 = production_digest(fx.CANONICAL_OFFER, CHAIN_ID, SETTLEMENT)
d2 = production_digest(salted, CHAIN_ID, SETTLEMENT)
print("0x" + d1.hex())
print("0x" + d2.hex())
print("bytes that differ:", sum(a != b for a, b in zip(d1, d2)), "of 32")
```

Essentially all of them (~31–32 on average). keccak is an avalanche: one changed input
bit flips about half of all output bits. That's why the digest works as an identity —
there is no such thing as "a slightly different offer" at the digest level; any change,
anywhere, is a completely different promise. (And why 03's `consumed` ledger, keyed by
digest, can't be dodged by tweaking a field: tweaking a field forges a *new* offer that
Bell never signed.)

</details>

**✏️ Your turn 3 — break the dynamic-bytes rule**

The rule that bites everyone: `params` enters the grid as `keccak(params)`, never raw.
Rebuild the layer-2 grid but "forget" the rule — pass `params` through `eth_abi.encode`
as type `"bytes"` instead of hashing it to a `"bytes32"` — then carry your wrong grid
through layers 3 and 5. Compare your wrong digest to the production one. (You're
reproducing the single most common real-world EIP-712 integration bug.)

In [ ]:
# wrong_encoded = eth_abi.encode([...types with "bytes"...], [...raw params...])
# wrong_struct  = keccak(wrong_encoded)
# wrong_digest  = keccak(b"\x19\x01" + domain_separator + wrong_struct)
# ...compare with the production digest...

<details><summary>✅ Solution 3 — peek only after trying</summary>

```python
types = ["bytes32", "address", "address", "uint8", "bytes32",
         "bytes",                                   # ← the mistake
         "uint64", "uint64", "address", "uint256", "uint64", "bytes32", "bytes32"]
values = [OFFER_TYPEHASH, wire["provider"], wire["consumer"], wire["serviceType"],
          bytes.fromhex(wire["resourceId"][2:]),
          bytes.fromhex(wire["params"][2:]),        # raw, unhashed
          wire["startTime"], wire["endTime"], wire["paymentToken"],
          int(wire["price"]), wire["validUntil"],
          bytes.fromhex(wire["salt"][2:]), bytes.fromhex(wire["termsHash"][2:])]
wrong_digest = keccak(b"\x19\x01" + domain_separator + keccak(eth_abi.encode(types, values)))
print("wrong :", "0x" + wrong_digest.hex())
print("right :", "0x" + prod.hex())
print("equal?", wrong_digest == prod)
```

Different digest — so a signature built this way recovers to a stranger at the
contract, and every fulfill dies at `BadSignature` even though *both sides are honest
and agree on every field*. This failure mode is why "must match byte-for-byte" is
written at the top of `chainmcp/src/chainmcp/signing.py`, and why the repo keeps a
cross-stack test whose only job is comparing the Python digest to the contract's.

</details>

## 4 · The contract as referee

Everything so far was Python agreeing with Python. The real test: the **contract**
computes its own digest, in Solidity, from the same offer — `hashOffer` is a public
function precisely so tools can ask for ground truth instead of re-deriving it. Time to
put our hand-built bytes in front of the referee. (Same lab pattern as chapter 03:
disposable chain, skip politely if Foundry's missing.)

In [ ]:
from chainmcp.testing import anvil_available, artifacts_available, launch_anvil
from chainmcp.client import ChainClient, ChainRevert

CHAIN_OK = anvil_available() and artifacts_available()
SKIP = ("skipped: needs anvil + built contracts — install Foundry "
        "(https://getfoundry.sh), then run:  forge build --root contracts")

anvil = ada = bell = None
print("anvil on PATH   →", "✓" if anvil_available() else "✗")
print("forge artifacts →", "✓" if artifacts_available() else "✗")
if not CHAIN_OK:
    print(SKIP)

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    from web3 import Web3
    from chainmcp.artifacts import find_contracts_dir, load_abi

    anvil = launch_anvil(timestamp=fx.WINDOW.start - 900)          # 13:45, story time
    print("deployed at:", anvil.deployment["A2ASettlement"])
    print("our layer-4 constant:", SETTLEMENT)
    print("deterministic deployment held?",
          anvil.deployment["A2ASettlement"] == SETTLEMENT)

    w3 = Web3(Web3.HTTPProvider(anvil.rpc_url))
    settlement = w3.eth.contract(
        address=anvil.deployment["A2ASettlement"],
        abi=load_abi("A2ASettlement", find_contracts_dir()),
    )

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    # The offer as the CONTRACT wants it: a tuple of the twelve fields, in struct order.
    offer_tuple = (
        wire["provider"], wire["consumer"], wire["serviceType"],
        bytes.fromhex(wire["resourceId"][2:]), bytes.fromhex(wire["params"][2:]),
        wire["startTime"], wire["endTime"], wire["paymentToken"],
        int(wire["price"]), wire["validUntil"],
        bytes.fromhex(wire["salt"][2:]), bytes.fromhex(wire["termsHash"][2:]),
    )
    contract_digest = settlement.functions.hashOffer(offer_tuple).call()

    print("the contract's hashOffer :", "0x" + contract_digest.hex())
    print("your hand-built digest   :", "0x" + digest.hex())
    print("Solidity == your Python? ", contract_digest == digest)

Pause on that. A Solidity contract, running keccak in the EVM over `abi.encode` of a
calldata struct, and your notebook cells running `eth_abi` + `keccak` by hand, produced
**the same 32 bytes**. That equality is the entire trust story of the offer pipeline:
*what is signed is exactly what is verified*. There is no translation step left to
trust, because both sides computed the same value independently.

And therefore the production signature — chapter 03 used `bell.sign_offer(...)` as
magic — should be exactly our hand signature:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    ada  = ChainClient(anvil.rpc_url, ANVIL_KEYS["ada"],  deployment=anvil.deployment)
    bell = ChainClient(anvil.rpc_url, ANVIL_KEYS["bell"], deployment=anvil.deployment)

    production_signed = bell.sign_offer(fx.CANONICAL_OFFER, terms_doc=fx.TERMS_DOC)

    print("chainmcp's signature :", production_signed.signature[:40] + "…")
    print("your signature       :", "0x" + signed_digest.signature.hex()[:38] + "…")
    print("identical 65 bytes?  ", production_signed.signature == "0x" + signed_digest.signature.hex())

Same digest, same key, same 65 bytes — the black box is now empty; you *are* the
signer. Last: replay chapter 03's tamper robbery, knowing now exactly which layer
catches it. Lower the price after Bell signed; the contract rebuilds the digest from
the offer *it received* (layers 1–5, in Solidity), recovers the signer from your
signature against *that* digest — and gets a stranger:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    from a2a_interfaces.models import SignedOffer

    ada.faucet(int(fx.PRICE_10_TOK))
    discounted = fx.CANONICAL_OFFER.model_copy(update={"price": "9000000000000000000"})
    tampered = SignedOffer(offer=discounted, signature=production_signed.signature,
                           terms_doc=fx.TERMS_DOC)
    try:
        ada.approve_and_fulfill(tampered)
    except ChainRevert as e:
        print("tampered offer →", e)

    tx, ticket = ada.approve_and_fulfill(production_signed)     # the honest one still works
    print("honest offer   → ticket", ticket, "minted to", ada.owner_of(ticket)[:10] + "…")

**✏️ Your turn 4 — the signature that's worthless elsewhere**

Layer 4 claimed domain binding kills cross-chain replay. Prove it without any second
chain: compute the production digest of the canonical offer for **Ethereum mainnet**
(`chain_id=1`, same settlement address) and compare it to the anvil-domain digest.
Then answer in a comment: Mallory steals Bell's 65-byte signature from our lab chain —
what exactly can he do with it on mainnet?

In [ ]:
# mainnet_digest = production_digest(fx.CANONICAL_OFFER, ..., SETTLEMENT)
# ...compare to `prod` (the 31337 digest)...
# Mallory's stolen signature is useful on mainnet for: ...

<details><summary>✅ Solution 4 — peek only after trying</summary>

```python
mainnet_digest = production_digest(fx.CANONICAL_OFFER, 1, SETTLEMENT)
print("chain 31337:", "0x" + prod.hex())
print("chain 1    :", "0x" + mainnet_digest.hex())
print("equal?", mainnet_digest == prod)
```

Different digests — the chain id sits inside the domain separator, so "the same offer
on mainnet" is a different 32-byte promise. Bell's stolen signature recovers to Bell
only against the *lab* digest; against the mainnet digest it recovers to a stranger →
`BadSignature`. The stolen bytes are worth exactly nothing anywhere but the one
contract, on the one chain, they were made for.

</details>

In [ ]:
# Always clean up your disposable world — never orphan a chain process.
if anvil is not None:
    for client in (ada, bell):
        client.close()
    anvil.stop()
    print("world ended. (It was disposable — that was the point.)")
else:
    print("nothing to clean up")

## 5 · Whose key, whose promise — signing as inventory commitment

Two closing ideas turn this mechanism into *architecture*.

**The signature is Bell's inventory commitment.** Notice what the system deliberately
does *not* have: a reservations database that "holds" capacity when Bell quotes. It
doesn't need one, because the entitlement mints **at settlement, from the signed offer
itself** — so the moment Bell's key touches an offer, that offer *is* sellable
inventory: anyone holding it can turn it into a binding on-chain ticket before
`validUntil`, and no later step asks Bell's opinion. The discipline this forces is
one sentence long, and it's in the paper (§4.4): **he signs only what he can honor.**
Where does "what he can honor" get decided? By deterministic arithmetic over
already-sold capacity — Bell's capacity ledger, at quote time. You'll build it in
chapter 07b, and it's the reason one probe of the adversarial matrix is *allowed* by
design (a second valid ticket for the same resource mints fine — overselling was
prevented before signing, not after).

**Where do keys live? In exactly one room.** This notebook handled `bell_key` directly
because it's a lab constant on a disposable chain. The real system never does — and
that's a hard rule of the repo (rule 2), not a habit:

> **🧭 Decision (principled) — keys live only in the custodian; agents hold none**
>
> **Chosen:** each agent has a *key custodian* — `chainmcp`, an MCP tool server — and
> it is the **only** process that ever touches a private key. The LangGraph agent asks
> it "sign this offer" and receives 65 bytes; the controller *verifies* signatures all
> day and can sign **nothing** (it holds no key at all).
> **Alternatives:** (a) the agent process holds its own key — simplest, one process
> fewer; (b) one shared signing service for everybody.
> **Why:** (a) puts a spending key inside the same process as LLM-driven code — the
> component that ingests untrusted text (offers, A2A messages) and acts on model
> output. One prompt injection, one jailbreak, one buggy tool call, and the key is
> exfiltrated or misused. The custodian boundary means the worst a hijacked agent can
> do is *request signatures* — visible, rate-limitable, policy-checkable — never leak
> the secret itself. (b) concentrates both parties' trust in one box, which is the
> middleman of chapter 03 wearing a new hat. And the controller not-signing matters:
> a component that only verifies can be audited as pure logic; nothing it computes can
> move money.
> **Cost:** an extra process per agent and a network hop per signature — cheap next to
> the failure mode it removes.
> **In the paper:** §4.2 (each agent "holding no keys"; the custodian) and §4.6, where
> *Keys* is the first of the three structural trust boundaries.

## 6 · What you can now say

- **What a signature is:** 65 bytes produced with a secret key over a message, from
  which anyone can recover the signer's address — and the recovery turns to a stranger
  if one byte of the message changes. You signed, recovered, and tampered.
- **Why "just sign the offer" is naive:** you ran both ambiguity robberies —
  same-offer-different-bytes (Robbery A) and different-offers-same-bytes (Robbery B).
  Canonical, typed encoding is not pedantry; it's the difference between a promise and
  a pun.
- **What EIP-712 actually does:** typehash (shape) → fixed 32-byte grid with dynamic
  fields hashed (values) → struct hash (identity) → domain separator (place) → final
  digest. You built all five layers and matched the contract's own `hashOffer` — and
  chainmcp's production signer — byte for byte.
- **Why the salt/consumed ledger from 03 works:** the digest is an avalanche-hash
  identity; any field change is a wholly new promise Bell never signed.
- **What domain binding buys:** a signature is a promise *to this contract, on this
  chain* — stolen bytes are worthless elsewhere. You computed the mainnet digest and
  watched it differ.
- **Why keys live in one room:** the custodian boundary keeps spending keys out of
  LLM-adjacent processes; the controller verifies but can never sign.

**Loose threads, on purpose:**

- Bell's offer is signed — but how does Ada later prove *to the controller* that she
  owns ticket #7? Same primitive, different ceremony: a challenge–response with a
  single-use nonce (and an EIP-191 `personal_sign`, the simpler cousin you used in §1).
  That's **chapter 05**, where the bouncer finally gets built.
- "He signs only what he can honor" — the capacity ledger that enforces it is
  **chapter 07b**.

*Next: [05 — The bouncer](05_the_bouncer.ipynb)*

## 7 · 📝 For the paper

Where this chapter lands (`main-arxiv.tex` coordinates): the signature half of **§4.4
(smart-contract and NFT lifecycle)** — the offer struct, EIP-712, single-use digests —
and the *Keys* boundary of **§4.6 (trust boundaries)**; the forged-signature row of the
adversarial matrix (**§6.2 → §7.2**). Draft sentences you can defend, each with the
evidence you personally ran:

| you can write… | because you ran… |
|---|---|
| *A sale begins as an EIP-712 offer struct signed by the provider's key; what is signed is exactly what the contract verifies, byte for byte.* | your five hand-built layers `==` the contract's `hashOffer` `==` chainmcp's production digest (§3–§4) |
| *Because the entitlement is minted from the signed offer at settlement, signing is the provider's inventory commitment: he signs only what he can honor.* | §5's argument + the tamper robbery — nothing after the signature asks Bell's opinion |
| *Signed offers are single-use: the contract keys its consumed-ledger by the offer's EIP-712 digest, so any modification is not a variant but a different, unsigned promise.* | Your turn 2 — one salt byte flipped ~all 32 digest bytes |
| *The domain separator binds every signature to one contract on one chain, so signatures cannot be replayed across deployments.* | Your turn 4 — chain id 1 vs 31337 produced disjoint digests |
| *Private keys live in exactly one component per agent — its custodian; the controller verifies signatures but never signs.* | §5's decision box (design argument; the enforcing rule is repo rule 2) |

**Reviewer objections you can now answer from experience:** *"Why EIP-712 and not
signed JSON?"* — you broke signed JSON two ways in §2 without any attacker. *"What
about signature malleability?"* — the replay defense is the digest-keyed consumed
ledger (a mauled signature still signs the same digest — 03 §4), with OpenZeppelin's
malleability rejection as belt on top. *"Could a stolen signature be replayed on
another chain?"* — Your turn 4.

**Honesty inventory for §8.4:** the cryptographic primitives are entirely standard
(secp256k1, keccak-256, OpenZeppelin `ECDSA.recover`) — the contribution is the seam
discipline (three codebases agreeing on the bytes), not the cryptography; and in the
prototype the "custodian" runs on the same machine as the agents (§5.5's co-located
testbed), so the key-isolation argument is architectural, not yet operationally
hardened (no HSM, no rate limiting).

*Next: [05 — The bouncer](05_the_bouncer.ipynb)*